In [10]:
import os
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, OpenAIChatCompletionsModel, trace, function_tool

In [2]:
load_dotenv(override=True)

True

In [3]:
# Create Groq client (OpenAI-compatible)
client = AsyncOpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

In [4]:
# Attach model
model = OpenAIChatCompletionsModel(
    model="openai/gpt-oss-20b",   # Groq model name
    openai_client=client,
)

In [ ]:
agent = Agent(
    name="GroqAgent",
    instructions="You are a helpful AI assistant",
    model=model,
)

In [6]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [7]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=model
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=model
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=model
)

In [8]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

**Subject:** Cut Your SOC 2 Audit Time by 70% with AI  

Hi [First Name],

I know SOC 2 audits can feel like a full‑time job—manual evidence gathering, endless spreadsheets, and the constant worry that something’s missing.  

At **ComplAI** we turn that burden into a streamlined, AI‑powered workflow. Our platform automatically maps your controls to the SOC 2 framework, generates real‑time evidence, and flags gaps before the auditors even call.  

Key benefits for teams like yours:  

| What you’ll gain | How it happens | Why it matters |
|------------------|----------------|----------------|
| **70 % faster audit prep** | Continuous control monitoring & auto‑generation of evidence | Get audit‑ready in weeks instead of months |
| **Reduced audit costs** | One‑click evidence bundles & risk heat maps | Cut legal & consulting spend |
| **Clear risk visibility** | AI‑driven insights & priority scoring | Focus on high‑impact issues first |

Would you have 20 minutes next week to see how we c

In [11]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

**Subject:** Automate SOC 2 Compliance & Audit Readiness with AI

Hi [First Name],

I hope this note finds you well. In today’s regulatory landscape, keeping SOC 2 compliant while scaling operations can consume 30 % of an IT team’s time – and a single lapse can trigger costly audit findings.

**ComplAI** is an AI‑powered SaaS platform that:

- **Automates evidence collection** across all controls, slashing manual effort by up to 70 %.  
- **Generates audit‑ready reports** in minutes, with real‑time compliance dashboards.  
- **Detects gaps proactively**, providing actionable remediation recommendations.  
- **Scales effortlessly** with cloud workloads and hybrid environments.

Our clients have seen audit preparation time drop from weeks to days and have avoided $250 k in remediation costs in the first year.

Could we schedule a 15‑minute demo next week to explore how ComplAI can fit your current compliance workflow? I’ll send a calendar invite.

Thank you for your time,  
[Your Name]  

In [12]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=model
)

In [13]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
**Subject:** Accelerate Your SOC 2 Compliance Without the Overhead

Hi [First Name],

I hope your week is going well. I’m reaching out because I know how demanding SOC 2 compliance can be for teams that are already stretched thin on product development, customer support, and security operations.

**The challenge**  
- Manual policy creation and continuous monitoring can take weeks of effort and constant coordination between security, engineering, and compliance leads.  
- Even with the best tooling, the audit preparation process often ends up as a firefighting exercise, leaving teams scrambling for evidence at the last minute.

**How ComplAI changes the game**  
- **AI‑driven policy generation**: Our engine auto‑creates SOC 2 controls that align with your actual tooling stack and workflow.  
- **Continuous evidence collection**: Pull real‑time logs, configurations, and activity feeds—no more manual export‑and‑re‑import.  
- **Audit‑ready dashboards**: Instantly see au

### Create tool to send email

In [17]:
@function_tool
def send_email(body: str):
    """Send email using this tool"""
    return {"status": "sucess"}

In [18]:
send_email

FunctionTool(name='send_email', description='Send email using this tool', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000026172B0B6A0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None)

In [20]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools


[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function Agent.as_tool.<locals>._run_agent_tool at 0x0000026176590860>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function Agent.as_tool.<locals>._run_ag

In [22]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=model)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)